Hi! I am a Kaggle beginner and not fluent in English, so this notebook is written in Japanese. 

Please use your browser's translation (or DeepL/ChatGPT) to read it!

Hope it helps!

# これまでの進捗（簡易版）

**ベスト CV スコア（LogLoss） = 0.3993**

- v1 : 数値列 + 決定木のもっとも単純なモデルで、Kaggle 初提出を済ませる
- v2 : ProfileReport を用いた、探索的データ分析（EDA）
- v3 : 特徴量エンジニアリング
- v5 : LightGBM（勾配ブースティング）
- v6 : LightGBM、ハイパーパラメータの調節（num_leaves）
- v7 : 交差検証
- v8 : v3 の特徴量エンジニアリングを LightGBM に反映
- v9 : 閾値の再設定
- v10 : Fare_per_person, Title_id（Name 列から敬称抽出）を追加
- v12 : Age の欠損値を手動で補完
- v13 : Optuna によるハイパーパラメータのチューニング

# 今回の方針

アンサンブルを試してみる。

今回は以下の 3 種類のモデルを混ぜる。

1. LightGBM
2. ランダムフォレスト（過学習に強い）
3. ロジスティック回帰（シンプルな線形モデル）

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import log_loss

In [ ]:
import kagglehub

path = kagglehub.competition_download('titanic')

print("Path to competition files:", path)

In [ ]:
train = pd.read_csv("../input/competitions/titanic/train.csv")
test = pd.read_csv("../input/competitions/titanic/test.csv")
gender_submission = pd.read_csv("../input/competitions/titanic/gender_submission.csv")

In [ ]:
train.head()

- PassengerId : 乗客ID
- Survived : 生存状況（0 = 死亡, 1 = 生存） **※ train のみ**
- Pclass : チケットクラス（1 = 上位, 2 = 中位, 3 = 下位）
- Name : 乗客名
- Sex : 性別
- Age : 年齢
- SibSp : 同乗していた兄弟姉妹／配偶者の数
- Parch : 同乗していた親／子の数
- Ticket : チケット番号
- Fare : 運賃
- Cabin : 客室番号
- Embarked : 乗船港

# データの前処理

In [ ]:
# SettingWithCopyWarning エラー対策
# 元データへの参照（ビュー）を断ち切り、メモリ上に独立した新しいデータを作成
train = train.copy()
test = test.copy()

categorical_features = ["Embarked", "Pclass", "Sex"]

# カテゴリ変数を category 型に変換
for col in categorical_features:
    train[col] = train[col].astype('category')
    test[col] = test[col].astype('category')

In [ ]:
train["FamilySize"] = train["SibSp"] + train["Parch"] + 1
test["FamilySize"] = test["SibSp"] + test["Parch"] + 1

train["Fare_per_person"] = train["Fare"] / train["FamilySize"]
test["Fare_per_person"] = test["Fare"] / test["FamilySize"]

In [ ]:
# Name から敬称を抽出
train["Title"] = train["Name"].str.extract(r" ([A-Za-z]+)\.", expand=False)
test["Title"] = test["Name"].str.extract(r" ([A-Za-z]+)\.", expand=False)

train["Title"] = train["Title"].astype("category")
test["Title"] = test["Title"].astype("category")

In [ ]:
# train データを使って「敬称ごとの年齢中央値」を計算
title_age_medians = train.groupby("Title", observed=False)["Age"].median()

# 年齢の欠損値を敬称別の中央値で補完
train["Age_imp"] = train["Age"].fillna(train["Title"].map(title_age_medians))
test["Age_imp"] = test["Age"].fillna(test["Title"].map(title_age_medians))

# 全体の中央値で残りの欠損を埋める（test 側で未知の敬称が出た際の保険）
train["Age_imp"] = train["Age_imp"].fillna(train["Age"].median())
test["Age_imp"] = test["Age_imp"].fillna(train["Age"].median())

In [ ]:
categorical_features = ["Embarked", "Pclass", "Sex"]

# モデルに使用する特徴量（Age を補完済みの Age_imp に変更）
col = [
    "Pclass",
    "Sex",
    "Age_imp",
    "Embarked",
    "FamilySize",
    "Fare_per_person",
]

X_train = train[col]
y_train = train["Survived"]

X_test = test[col]

# アンサンブルの実行

In [ ]:
# 各モデルの予測結果を保持
preds_lgb = []
preds_rf = []
preds_lr = []

In [ ]:
callbacks = [
    lgb.early_stopping(stopping_rounds=10),
    lgb.log_evaluation(period=10)
]

params = {
    "objective": "binary",
    "num_leaves": 5
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

In [ ]:
# Scikit-learn モデル（RF/LR）用にカテゴリ変数をダミー変数化（One-Hot Encoding）
X_train_encoded = pd.get_dummies(X_train, drop_first=True)
X_test_encoded = pd.get_dummies(X_test, drop_first=True)

# train と test の列順・列数を揃える
X_train_encoded, X_test_encoded = X_train_encoded.align(
    X_test_encoded, join="left", axis=1, fill_value=0
)

# 欠損値補完（ロジスティック回帰のエラー防止用）
X_train_encoded = X_train_encoded.fillna(0)
X_test_encoded = X_test_encoded.fillna(0)

# ロジスティック回帰用の標準化
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_encoded)
X_test_scaled = scaler.transform(X_test_encoded)

In [ ]:
for fold_id, (train_index, valid_index) in enumerate(cv.split(X_train, y_train)):
    X_tr, y_tr = X_train.iloc[train_index], y_train.iloc[train_index]
    X_val, y_val = X_train.iloc[valid_index], y_train.iloc[valid_index]

    # --- 1. LightGBM ---
    model_lgb = lgb.LGBMClassifier(**params, n_estimators=300)
    model_lgb.fit(
        X_tr,
        y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=callbacks,
    )
    preds_lgb.append(model_lgb.predict_proba(X_test)[:, 1])

    # --- 2. Random Forest ---
    X_tr_enc = X_train_encoded.iloc[train_index]
    model_rf = RandomForestClassifier(
        n_estimators=100, max_depth=5, random_state=0
    )
    model_rf.fit(X_tr_enc, y_tr)
    preds_rf.append(model_rf.predict_proba(X_test_encoded)[:, 1])

    # --- 3. Logistic Regression ---
    X_tr_sc = X_train_scaled[train_index]
    model_lr = LogisticRegression(random_state=0)
    model_lr.fit(X_tr_sc, y_tr)
    preds_lr.append(model_lr.predict_proba(X_test_scaled)[:, 1])

# 5-Fold の平均を計算
y_pred_lgb = np.mean(preds_lgb, axis=0)
y_pred_rf = np.mean(preds_rf, axis=0)
y_pred_lr = np.mean(preds_lr, axis=0)

# 3つのモデルの予測結果を単純平均（アンサンブル）
y_pred_ensemble = (y_pred_lgb + y_pred_rf + y_pred_lr) / 3

In [ ]:
# ループの前に OOF 保持用配列を用意
oof_lgb = np.zeros(len(X_train))
oof_rf = np.zeros(len(X_train))
oof_lr = np.zeros(len(X_train))

In [ ]:
for fold_id, (train_index, valid_index) in enumerate(cv.split(X_train, y_train)):
    X_tr, y_tr = X_train.iloc[train_index], y_train.iloc[train_index]
    X_val, y_val = X_train.iloc[valid_index], y_train.iloc[valid_index]

    # --- 1. LightGBM ---
    model_lgb = lgb.LGBMClassifier(**params, n_estimators=300)
    model_lgb.fit(
        X_tr,
        y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=callbacks,
    )
    oof_lgb[valid_index] = model_lgb.predict_proba(X_val)[:, 1]
    preds_lgb.append(model_lgb.predict_proba(X_test)[:, 1])

    # --- 2, Random Forest ---
    X_tr_enc = X_train_encoded.iloc[train_index]
    X_val_enc = X_train_encoded.iloc[valid_index]
    model_rf = RandomForestClassifier(
        n_estimators=100, max_depth=5, random_state=0
    )
    model_rf.fit(X_tr_enc, y_tr)
    oof_rf[valid_index] = model_rf.predict_proba(X_val_enc)[:, 1]
    preds_rf.append(model_rf.predict_proba(X_test_encoded)[:, 1])

    # --- 3. Logistic Regression ---
    X_tr_sc = X_train_scaled[train_index]
    X_val_sc = X_train_scaled[valid_index]
    model_lr = LogisticRegression(random_state=0)
    model_lr.fit(X_tr_sc, y_tr)
    oof_lr[valid_index] = model_lr.predict_proba(X_val_sc)[:, 1]
    preds_lr.append(model_lr.predict_proba(X_test_scaled)[:, 1])

# OOF でのアンサンブル予測
oof_ensemble = (oof_lgb + oof_rf + oof_lr) / 3

In [ ]:
# 各モデルとアンサンブルの CV スコア（LogLoss）を計算
score_lgb = log_loss(y_train, oof_lgb)
score_rf = log_loss(y_train, oof_rf)
score_lr = log_loss(y_train, oof_lr)
score_ens = log_loss(y_train, oof_ensemble)

print(f"LightGBM    CV (LogLoss): {score_lgb:.4f}")
print(f"RandomForest CV (LogLoss): {score_rf:.4f}")
print(f"Logistic Reg CV (LogLoss): {score_lr:.4f}")
print(f"----------------------------------------")
print(f"Ensemble     CV (LogLoss): {score_ens:.4f}")

# バーチャートでスコアを比較可視化
models = ["LightGBM", "RandomForest", "LogisticReg", "Ensemble"]
scores = [score_lgb, score_rf, score_lr, score_ens]
colors = ["skyblue", "lightgreen", "coral", "gold"]

plt.figure(figsize=(8, 5))
bars = plt.bar(models, scores, color=colors)
plt.ylabel("LogLoss (Lower is better)")
plt.title("Model CV Score Comparison")
plt.ylim(min(scores) - 0.02, max(scores) + 0.02)

# バーの上に数値を表示
for bar in bars:
    yval = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        yval + 0.002,
        f"{yval:.4f}",
        ha="center",
        va="bottom",
    )

plt.show()

# 考察

今回はアンサンブルしない方が良い。

最初に雑にアンサンブルして、どのモデルを主力にするかを決めるのはあり。

# 提出

今回の CV スコア = 0.3954

In [ ]:
# 5 つのモデルの予測値の平均をとる（アンサンブル）
y_pred_avg = np.mean(y_preds, axis=0)

y_pred_binary = (y_pred_avg > 0.5).astype(int)

y_pred_binary[:10]

In [ ]:
sub = gender_submission.copy()
sub["Survived"] = list(map(int, y_pred_binary))
sub.to_csv("submission.csv", index=False)